In [1]:
import os
import torch
import torch.nn.functional as F
import itertools
from itertools import cycle
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from diffusers import StableDiffusionPipeline, T2IAdapter
from controlnet_aux import OpenposeDetector
from tqdm import tqdm

REPO_ROOT = Path().absolute().parent
DEVICE = "cuda"
LAMBDA = 1.0
NUM_TRAIN = 5
LR = 5e-6
BATCH_SIZE = 1
UNIQUE_TOKEN = "sks"

INSTANCE_BASE_DIR = str(REPO_ROOT / "datasets" / "dataset")
PRIOR_BASE_DIR = str(REPO_ROOT / "datasets" / "dataset_prior")
MODEL_SAVE_DIR = str(REPO_ROOT / "outputs" / "saved_models" / "our_controlnet")

CLASSES_DICT = {
    'humanoid robot': ['unitree']
}

class PoseImageDataset(Dataset):
    def __init__(self, data_dir, detector):
        self.image_paths = [os.path.join(data_dir, f) for f in os.listdir(data_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
        self.detector = detector
        self.image_transforms = transforms.Compose([
            transforms.Resize((512, 512), interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]),
        ])
        self.cond_transforms = transforms.Compose([
            transforms.Resize((512, 512), interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.ToTensor(),
        ])
        
    def __len__(self): 
        return len(self.image_paths)
        
    def __getitem__(self, idx): 
        img = Image.open(self.image_paths[idx]).convert("RGB")
        pose = self.detector(img)
        return self.image_transforms(img), self.cond_transforms(pose)

def class_collate_fn(examples):
    pixel_values = torch.stack([ex[0] for ex in examples])
    pose_values = torch.stack([ex[1] for ex in examples])
    return pixel_values, pose_values

def train_and_save(class_name, instance):
    instance_dir = os.path.join(INSTANCE_BASE_DIR, instance)
    class_dir = os.path.join(PRIOR_BASE_DIR, instance) 
    save_path = os.path.join(MODEL_SAVE_DIR, instance)
    
    os.makedirs(save_path, exist_ok=True)
    
    instance_prompt = f"a {UNIQUE_TOKEN} {class_name}"
    class_prompt = f"a {class_name}"
    
    detector = OpenposeDetector.from_pretrained("lllyasviel/ControlNet")
    adapter = T2IAdapter.from_pretrained("TencentARC/t2iadapter_openpose_sd14v1", torch_dtype=torch.float32).to(DEVICE)
    pipeline = StableDiffusionPipeline.from_pretrained("runwayml/stable-diffusion-v1-5", torch_dtype=torch.float32, safety_checker=None).to(DEVICE)
    vae, text_encoder, tokenizer, unet, noise_scheduler = pipeline.vae, pipeline.text_encoder, pipeline.tokenizer, pipeline.unet, pipeline.scheduler
    
    vae.requires_grad_(False).eval()
    adapter.requires_grad_(False).eval()
    text_encoder.requires_grad_(True).train()
    unet.requires_grad_(True).train()
    
    optimizer = torch.optim.AdamW(itertools.chain(unet.parameters(), text_encoder.parameters()), lr=LR, weight_decay=1e-2)
    
    inst_loader = DataLoader(PoseImageDataset(instance_dir, detector), batch_size=BATCH_SIZE, shuffle=True, collate_fn=class_collate_fn)
    class_loader = DataLoader(PoseImageDataset(class_dir, detector), batch_size=BATCH_SIZE, shuffle=True, collate_fn=class_collate_fn)
    inst_iter, class_iter = cycle(inst_loader), cycle(class_loader)

    def tokenize(prompt): 
        return tokenizer(prompt, padding="max_length", max_length=tokenizer.model_max_length, truncation=True, return_tensors="pt").input_ids.to(DEVICE)
    
    inst_tokens, class_tokens = tokenize(instance_prompt), tokenize(class_prompt)

    for step in tqdm(range(1, NUM_TRAIN + 1)):
        optimizer.zero_grad()
        
        inst_pixels, inst_poses = next(inst_iter)
        class_pixels, class_poses = next(class_iter)
        
        pixels = torch.cat([inst_pixels, class_pixels]).to(DEVICE)
        poses = torch.cat([inst_poses, class_poses]).to(DEVICE)
        
        with torch.no_grad():
            latents = vae.encode(pixels).latent_dist.sample() * vae.config.scaling_factor
            adapter_state = adapter(poses)
            
        prompt_embeds = torch.cat([text_encoder(inst_tokens)[0], text_encoder(class_tokens)[0]])
        noise = torch.randn_like(latents)
        timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (latents.shape[0],), device=DEVICE).long()
        noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)
        
        noise_pred = unet(noisy_latents, timesteps, encoder_hidden_states=prompt_embeds, down_block_additional_residuals=[state for state in adapter_state]).sample
        noise_pred_inst, noise_pred_prior = noise_pred.chunk(2, dim=0)
        target_inst, target_prior = noise.chunk(2, dim=0)
        
        loss = F.mse_loss(noise_pred_inst, target_inst) + LAMBDA * F.mse_loss(noise_pred_prior, target_prior)
        loss.backward()
        optimizer.step()

    unet.save_pretrained(os.path.join(save_path, "unet"))
    text_encoder.save_pretrained(os.path.join(save_path, "text_encoder"))
    del optimizer, inst_loader, class_loader, pipeline, adapter, detector
    torch.cuda.empty_cache()

if __name__ == "__main__":
    for class_name, instances in CLASSES_DICT.items():
        for instance in instances:
            train_and_save(class_name, instance)

/usr/local/lib/python3.12/dist-packages/controlnet_aux/mediapipe_face/mediapipe_face_common.py:7: UserWarning: The module 'mediapipe' is not installed. The package will have limited functionality. Please install it using the command: pip install 'mediapipe'
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.12/dist-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/usr/local/lib/python3.12/dist-packages/controlnet_aux/segment_anything/modeling/tiny_vit_sam.py:654: UserWarning: Overwriting tiny_vit_5m_224 in registry with controlnet_aux.segmen

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /workspace/.hf_home/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. F

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]